# Urdu Orpheus — checkpoint listening demo
Run on **Kaggle with Internet and an NVIDIA GPU**. Enable the **HF_TOKEN** secret for this notebook (read access to your private model repository is enough), or set the `HF_TOKEN` environment variable.

Run cells 1–3, choose checkpoint indices in cell 4, then run cell 5. Edit `TEXT` to your Urdu sentence. To compare more checkpoints, rerun cells 4–5; to change only the sentence, edit `TEXT` and rerun cell 5. Each model loads in a fresh process, so GPU memory is released between checkpoints. The same seed and decoding settings apply to every selection.

This notebook downloads inference files only from the project's unpacked Hub snapshots, without preparing the dataset or resuming training. Checkpoints are LoRA adapters, so the recorded base model is downloaded too. WAV files and generation metadata are saved under `urdu-checkpoint-demo/audio/`. Listening results demonstrate speech quality; they are not an objective evaluation score.


## 1. Settings


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

HF_REPO_ID = "hassan-31x/orpheus-urdutts-checkpoints"
HF_REVISION = "main"  # A branch, tag, or commit; cell 3 resolves it to one immutable commit.
RUN_ID = "urdutts-r32-lr1e4-seed3407-epoch1"  # None lists every run, including smoke runs.
HF_SECRET_NAME = "HF_TOKEN"
REPO_URL = "https://github.com/hassan-31x/orhpeus-ft-urdutts.git"
REPO_REF = "main"  # Can be pinned to a project Git commit.

TEXT = "آج موسم بہت خوشگوار ہے۔ ہم اردو بولنا سیکھ رہے ہیں۔"
SEED = 3407
TEMPERATURE = 0.6
TOP_P = 0.95
REPETITION_PENALTY = 1.1
MAX_NEW_TOKENS = 1400  # If audio is cut short, shorten TEXT or increase within the trained context limit.
SPEAKER = None  # UrduTTS has no speaker IDs; keep None for this training run.

ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".").resolve()
DEMO = ROOT / "urdu-checkpoint-demo"
REPO = DEMO / "code"
ENV = DEMO / "env"
PYTHON = str(ENV / "bin" / "python")
DEMO.mkdir(parents=True, exist_ok=True)

for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
    if os.environ.get(name, "").strip():
        os.environ["HF_TOKEN"] = os.environ[name].strip()
        break
else:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
        if not isinstance(token, str) or not token.strip():
            raise ValueError("Empty secret")
        os.environ["HF_TOKEN"] = token.strip()
        del token
    except Exception:
        raise RuntimeError(
            "Enable the HF_TOKEN secret for THIS notebook in Kaggle Add-ons > Secrets, "
            "or set HF_TOKEN in the environment. Read access to the model repository is required."
        ) from None
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"
print("Demo folder:", DEMO)


## 2. Install the existing inference environment
Run once per fresh runtime. Dependencies are isolated from the notebook kernel; all model work uses that environment.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
print("Project commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
subprocess.run([sys.executable, str(REPO / "setup_kaggle.py"),
                "--venv", str(ENV), "--gpu"], cwd=REPO, check=True)

# Run short Hub operations in the same environment as inference.
def worker(source, *args):
    subprocess.run([PYTHON, "-c", source, *map(str, args)], cwd=REPO, check=True)


## 3. List saved checkpoints
Indices refer to exact adapter folders, including snapshot hashes. `adapter_final` entries may duplicate the last checkpoint. The Hub commit is pinned for both listing and downloading.


In [ ]:
CATALOG = DEMO / "checkpoints.json"
worker(r"""
import json, re, sys
from pathlib import Path, PurePosixPath
from huggingface_hub import HfApi
repo_id, revision, run_id, output = sys.argv[1:]
api = HfApi()
commit = api.model_info(repo_id, revision=revision).sha
files = set(api.list_repo_files(repo_id, revision=commit))
folders = []
for filename in sorted(files):
    if PurePosixPath(filename).name != "adapter_config.json":
        continue
    folder = str(PurePosixPath(filename).parent)
    parts = PurePosixPath(folder).parts
    if run_id and not folder.startswith(f"runs/{run_id}/"):
        continue
    if f"{folder}/run_identity.json" not in files:
        continue
    weights = [f for f in files if str(PurePosixPath(f).parent) == folder
               and PurePosixPath(f).name.startswith("adapter_model")
               and f.endswith((".safetensors", ".bin"))]
    if not weights:
        continue
    step = next((int(m.group(1)) for p in reversed(parts)
                 if (m := re.fullmatch(r"checkpoint-(\d+)", p))), None)
    folders.append(dict(folder=folder, step=step, final=parts[-1] == "adapter_final"))
folders.sort(key=lambda x: (x["folder"].split("/snapshots/")[0],
                            x["step"] if x["step"] is not None else float("inf"), x["folder"]))
Path(output).write_text(json.dumps(dict(commit=commit, checkpoints=folders), indent=2))
""", HF_REPO_ID, HF_REVISION, RUN_ID or "", CATALOG)
catalog = json.loads(CATALOG.read_text())
CHECKPOINTS = catalog["checkpoints"]
HUB_COMMIT = catalog["commit"]
print("Hub commit:", HUB_COMMIT)
if not CHECKPOINTS:
    raise RuntimeError("No unpacked adapters found. Check RUN_ID (use None for all runs). "
                       "Legacy .tar.gz snapshots must be restored separately before inference.")
for i, item in enumerate(CHECKPOINTS):
    print(f"[{i}] {item['folder']}")


## 4. Select and download adapters
Set `SELECT` to indices printed above, for example `[0, 2, 4]`. The default selects the latest numbered checkpoint in the chosen run. You can also paste exact folders into `ADAPTER_FOLDERS` to bypass index selection. Only the chosen adapters, tokenizer files and run identities are downloaded.


In [ ]:
numbered = [i for i, item in enumerate(CHECKPOINTS) if item["step"] is not None]
SELECT = [max(numbered, key=lambda i: CHECKPOINTS[i]["step"])] if numbered else [len(CHECKPOINTS) - 1]
# SELECT = [0, 2]  # Uncomment and choose your checkpoint indices.
ADAPTER_FOLDERS = []  # Optional: ["runs/.../snapshots/checkpoint-.../checkpoint-..."]

if ADAPTER_FOLDERS:
    folders = list(dict.fromkeys(ADAPTER_FOLDERS))
else:
    if not SELECT or any(type(i) is not int or not 0 <= i < len(CHECKPOINTS) for i in SELECT):
        raise ValueError("SELECT must contain valid indices from cell 3")
    folders = list(dict.fromkeys(CHECKPOINTS[i]["folder"] for i in SELECT))
known = {item["folder"] for item in CHECKPOINTS}
if any(folder not in known for folder in folders):
    raise ValueError("Choose exact adapter folders listed in cell 3")
selection = DEMO / "selection.json"
selection.write_text(json.dumps(folders))
RESOLVED = DEMO / "resolved-adapters.json"
worker(r"""
import json, sys
from pathlib import Path, PurePosixPath
from huggingface_hub import HfApi, hf_hub_download
repo_id, commit, selection, output = sys.argv[1:]
files = HfApi().list_repo_files(repo_id, revision=commit)
resolved = []
for folder in json.loads(Path(selection).read_text()):
    local = None
    for filename in files:
        p = PurePosixPath(filename)
        if str(p.parent) != folder:
            continue
        name = p.name
        if not (name.startswith(("adapter_config", "adapter_model", "tokenizer"))
                or name in {"run_identity.json", "special_tokens_map.json", "added_tokens.json",
                            "vocab.json", "vocab.txt", "merges.txt"} or name.endswith(".model")):
            continue
        path = hf_hub_download(repo_id, filename, revision=commit)
        local = str(Path(path).parent)
    resolved.append(dict(folder=folder, local=local))
Path(output).write_text(json.dumps(resolved, indent=2))
""", HF_REPO_ID, HUB_COMMIT, selection, RESOLVED)
ADAPTERS = json.loads(RESOLVED.read_text())
for item in ADAPTERS:
    print("Ready:", item["folder"])


## 5. Generate and listen
Each player shows the same Urdu text synthesized with its selected checkpoint. Outputs include WAV, raw generated token IDs and decoding metadata. A warning flags truncated or malformed output; increase the token budget only within the recorded training context. Playback preserves the WAV amplitude.


In [ ]:
import hashlib
from IPython.display import Audio, display

OUTPUTS = DEMO / "audio"
OUTPUTS.mkdir(exist_ok=True)
results = []
for item in ADAPTERS:
    label = item["folder"]
    identity = json.loads((Path(item["local"]) / "run_identity.json").read_text())
    if bool(SPEAKER) != bool(identity["config"]["speaker_column"]):
        raise ValueError("SPEAKER must match this checkpoint's training configuration")
    tag = hashlib.sha256((HF_REPO_ID + HUB_COMMIT + label + TEXT + str(
        (SEED, TEMPERATURE, TOP_P, REPETITION_PENALTY, MAX_NEW_TOKENS, SPEAKER)
    )).encode()).hexdigest()[:12]
    output = OUTPUTS / f"sample-{tag}.wav"
    print("\nCheckpoint:", label, flush=True)
    print("Text:", TEXT, flush=True)
    command = [PYTHON, "-u", str(REPO / "synthesize.py"), "--adapter", item["local"],
               "--text", TEXT, "--output", str(output), "--seed", str(SEED),
               "--temperature", str(TEMPERATURE), "--top-p", str(TOP_P),
               "--repetition-penalty", str(REPETITION_PENALTY),
               "--max-new-tokens", str(MAX_NEW_TOKENS)]
    if SPEAKER:
        command += ["--speaker", SPEAKER]
    subprocess.run(command, cwd=REPO, check=True)
    metadata = json.loads(output.with_suffix(".json").read_text())
    metadata.update(hf_repo_id=HF_REPO_ID, hub_commit=HUB_COMMIT, adapter_folder=label)
    output.with_suffix(".json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
    if (metadata["reached_token_budget"] or not metadata["ended_with_speech_stop"]
            or metadata["invalid_frame_token_offset"] is not None):
        print("Warning: output may be incomplete or malformed; inspect the saved metadata.")
    print(f"{metadata['duration_seconds']:.2f}s audio | {metadata['elapsed_seconds']:.2f}s generation + decoding")
    display(Audio(filename=str(output), normalize=False))
    results.append(dict(checkpoint=label, text=TEXT, audio=str(output), metadata=metadata))
(OUTPUTS / "comparison.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
print("\nSaved comparison:", OUTPUTS / "comparison.json")
